# Debugging Prompts: Live API Evaluation

This Jupyter Notebook analyzes 5 poorly written prompts. For each scenario, we define the issues, provide an improved prompt applying prompt engineering best practices (role assignment, context, constraints, structure), and use the **Groq API** to generate live outputs representing the 'Before' and 'After' states.

In [16]:
%pip install -q requests

import os
import getpass
import requests
from IPython.display import display, Markdown

# Safely request API key via password input in the notebook
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    print("Please enter your Groq API key securely below:")
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key: ")

def get_groq_response(prompt, system_prompt=None):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
        "Content-Type": "application/json"
    }
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": messages,
        "temperature": 0.7
    }
    
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        return f"**API Error:** {response.status_code} - {response.text}"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Content Generation: The Vague Instruction

### Original Prompt
`Write an email to clients about our new feature.`

### Issues
- **Missing Role & Context:** Doesn't define who is sending the email or what the product/feature is.
- **Lack of Constraints:** No designated length, tone, or structure.
- **No Goal:** Doesn't tell the AI what the client should do after reading (Call to Action).

In [17]:
original_prompt_1 = "Write an email to clients about our new feature."

display(Markdown(f"**❌ Before Output (Original Prompt):**\n\n{get_groq_response(original_prompt_1)}"))

**❌ Before Output (Original Prompt):**

Here's an example email:

Subject: Introducing Our Latest Feature: [Feature Name]

Dear Valued Clients,

We are excited to announce the launch of our latest feature, [Feature Name], designed to enhance your experience and streamline your workflow. This innovative feature is now live and available for all clients to use.

**What is [Feature Name]?**

[Feature Name] is a [briefly describe the feature and its benefits]. This feature is the result of our ongoing efforts to improve and expand our services, and we are confident that it will make a significant impact on your business.

**Key Features:**

- [List the main features and benefits of the new feature]
- [Optional: Provide screenshots or videos to illustrate the feature]

**How to Access [Feature Name]**

To access [Feature Name], simply log in to your account and navigate to the [specific menu or tab]. If you need assistance or have any questions, please don't hesitate to reach out to our support team.

**Why Upgrade to [Feature Name]?**

By using [Feature Name], you can:

- [List the benefits of using the new feature, such as increased efficiency, improved accuracy, or enhanced collaboration]
- [Optional: Provide statistics or case studies to demonstrate the value of the feature]

**Get Started Today**

We invite you to explore [Feature Name] and experience the benefits for yourself. If you have any questions or need help getting started, please don't hesitate to contact us. We're here to support you every step of the way.

Thank you for your continued partnership and trust in our services.

Best regards,

[Your Name]
[Your Company Name]
[Contact Information]

Feel free to customize the email to fit your needs and style.

--- 
### Improved Prompt
Assigning persona, providing context, imposing constraints, and providing a clear CTA.
```text
Act as a Customer Success Manager at a B2B SaaS company. Write an email to our enterprise clients announcing our new 'Automated Reporting' feature.

Background: The feature saves 10 hours a week by automatically exporting dashboard metrics to PDF every Monday.
Tone: Professional, enthusiastic, yet concise.
Constraints: 
- Keep the email under 150 words.
- Use a bulleted list to highlight 2 key benefits.
- End with a strong Call to Action (CTA) inviting them to click a link to view a tutorial.
```

In [18]:
system_prompt_1 = "Act as a Customer Success Manager at a B2B SaaS company."
improved_prompt_1 = """Write an email to our enterprise clients announcing our new 'Automated Reporting' feature.

Background: The feature saves 10 hours a week by automatically exporting dashboard metrics to PDF every Monday.
Tone: Professional, enthusiastic, yet concise.
Constraints: 
- Keep the email under 150 words.
- Use a bulleted list to highlight 2 key benefits.
- End with a strong Call to Action (CTA) inviting them to click a link to view a tutorial."""

display(Markdown(f"**✅ After Output (Improved Prompt):**\n\n{get_groq_response(improved_prompt_1, system_prompt_1)}"))

**✅ After Output (Improved Prompt):**

Subject: Introducing Automated Reporting - Streamline Your Reporting Process

Dear Enterprise Clients,

We are excited to announce the launch of our new 'Automated Reporting' feature, designed to simplify and accelerate your reporting process. This innovative feature saves you 10 hours a week by automatically exporting dashboard metrics to PDF every Monday.

Key Benefits:

• **Save Time**: Focus on high-value tasks with our automated reporting, freeing up 10 hours a week to drive business growth.
• **Enhance Accuracy**: Reduce human error with automated reporting, ensuring consistent and reliable data exports.

To get started with Automated Reporting, click here to view our tutorial: [insert link]. In just a few minutes, you'll be able to unlock the full potential of this feature and take your reporting process to the next level.

Best regards,
[Your Name]
Customer Success Manager
[Your Company]

## 2. Programming: The Ambiguous Debug Request

### Original Prompt
`Why isn't this code working?\nfunction calc(a, b) { return a + c; }`

### Issues
- **Vague Problem Statement:** Fails to specify the programming language or what the expected behavior is.
- **No Role Assignment:** Doesn't prompt the AI to act as an expert, leading to a potentially superficial or passive answer.
- **Missing Context:** Real scenarios have much more surrounding context; a blind debug often results in hallucinations.

In [19]:
original_prompt_2 = "Why isn't this code working?\nfunction calc(a, b) { return a + c; }"

display(Markdown(f"**❌ Before Output (Original Prompt):**\n\n{get_groq_response(original_prompt_2)}"))

**❌ Before Output (Original Prompt):**

The code isn't working because it's trying to use a variable `c` that hasn't been defined. The function is named `calc` but it's supposed to add `a` and `b`, not `a` and `c`. 

Here's the corrected code:

```javascript
function calc(a, b) { 
  // return a + b; // If you want to return a+b 
  // return a + c; // This line is causing the error because c is not defined
  return a + b; // This is the correct way to add a and b
}
```

However, if you're trying to use a variable `c` from outside the function, you'll need to declare it before calling the `calc` function, or pass it as an argument to the function.

Here's an example of how you can declare `c` before calling the `calc` function:

```javascript
let c = 5; // declare c
function calc(a, b) { return a + c; }
console.log(calc(3, 5));
```

This code will output `8` because `c` is `5` and `a + b` is `8`.

If you're trying to pass `c` as an argument to the `calc` function, you would do something like this:

```javascript
function calc(a, b, c) { return a + c; }
console.log(calc(3, 5, 7));
```

This code will also output `10` because `a` is `3`, `c` is `7` and `a + c` is `10`.

--- 
### Improved Prompt
Adding role assignment, pinpointing the expected behavior and error context, and formatting as step-by-step instructions.
```text
Act as a Senior JavaScript Developer. Review the following JavaScript function meant to add two numbers together.

<code_snippet>
function calc(a, b) { return a + c; }
</code_snippet>

Task:
1. Identify the bug causing the ReferenceError.
2. Provide the corrected code in a code block.
3. Briefly explain the best practice for naming variables to avoid this in the future.
```

In [20]:
system_prompt_2 = "Act as a Senior JavaScript Developer."
improved_prompt_2 = """Review the following JavaScript function meant to add two numbers together.

<code_snippet>
function calc(a, b) { return a + c; }
</code_snippet>

Task:
1. Identify the bug causing the ReferenceError.
2. Provide the corrected code in a code block.
3. Briefly explain the best practice for naming variables to avoid this in the future."""

display(Markdown(f"**✅ After Output (Improved Prompt):**\n\n{get_groq_response(improved_prompt_2, system_prompt_2)}"))

**✅ After Output (Improved Prompt):**

**Bug Identification:**

The bug causing the ReferenceError is that the variable `c` is not defined within the function `calc`. The function is supposed to add two numbers, `a` and `b`, but it's using `c` instead of `b`.

**Corrected Code:**

```javascript
function calc(a, b) {
  return a + b;
}
```

**Best Practice:**

To avoid this type of bug in the future, follow these best practices:

1. **Use meaningful variable names**: Avoid using single-letter variable names like `c`. Instead, use descriptive names that indicate the purpose of the variable.
2. **Read the code from top to bottom**: When writing code, read it from top to bottom, making sure that each variable is defined before it's used.
3. **Use a linter**: A linter can help catch errors like this one by checking for undefined variables.
4. **Test your code**: Thoroughly test your code to catch errors like this one.

**Additional Advice:**

In this specific case, we can also consider using a more descriptive function name, such as `addTwoNumbers`, to make the code more readable and self-explanatory.

## 3. Data Analysis: The Unbounded Output

### Original Prompt
`Analyze this data and give me insights. Sales: Q1 ($50k), Q2 ($45k), Q3 ($60k), Q4 ($80k).`

### Issues
- **Overly Broad Objective:** "Give me insights" means different things to sales versus finance versus marketing.
- **No Formatting Constraints:** Output could be a single sentence or a 10-paragraph essay.
- **No Deep Analysis Mandated:** The AI might just regurgitate the numbers instead of looking for rates of change.

In [21]:
original_prompt_3 = "Analyze this data and give me insights. Sales: Q1 ($50k), Q2 ($45k), Q3 ($60k), Q4 ($80k)."

display(Markdown(f"**❌ Before Output (Original Prompt):**\n\n{get_groq_response(original_prompt_3)}"))

**❌ Before Output (Original Prompt):**

Based on the provided sales data, here are some insights:

1. **Seasonal Trend**: The sales data exhibits a seasonal trend, with a noticeable increase in sales from Q1 to Q4. This suggests that the business may be more successful during the holiday season (Q4).

2. **Quarterly Sales Growth**: 
   - Q1 to Q2: Sales decreased by $5,000 (10%).
   - Q2 to Q3: Sales increased by $15,000 (33.33%).
   - Q3 to Q4: Sales increased by $20,000 (33.33%).

The growth rates are not consistent across the quarters, which may indicate changes in market conditions, promotional activities, or other external factors.

3. **Total Sales**: The total sales for the year are $235,000.

4. **Average Quarterly Sales**: The average quarterly sales are $58,750.

5. **Comparison to Industry Benchmarks**: Without industry benchmarks, it's difficult to determine if the sales growth is above or below average. However, the overall sales increase from Q1 to Q4 suggests a positive trend.

6. **Recommendations**: To further optimize sales, consider:

   - Analyzing the Q2 sales dip to identify potential causes and implement corrective measures.
   - Focusing on promotional activities and marketing strategies during Q2 to boost sales.
   - Maintaining a strong presence in the market during Q4 to continue the growth trend.
   - Exploring opportunities to expand sales beyond the current seasonal trend.

These insights should help you better understand the sales data and identify areas for improvement.

--- 
### Improved Prompt
Setting a specialized role, enforcing a strict output structure, and providing an exact context limit.
```text
Act like a Financial Data Analyst. Review the quarterly sales data for the fiscal year:
Data: Q1 ($50k), Q2 ($45k), Q3 ($60k), Q4 ($80k).

Your task is to provide a brief executive summary. Please format the output strictly as follows:
- **Total Revenue:** [Sum of all quarters]
- **QoQ Growth:** [Highlight the largest percentage transition between consecutive quarters]
- **Key Takeaway:** [One sentence summarizing the trend for next year's forecasting]
```

In [22]:
system_prompt_3 = "Act like a Financial Data Analyst."
improved_prompt_3 = """Review the quarterly sales data for the fiscal year:
Data: Q1 ($50k), Q2 ($45k), Q3 ($60k), Q4 ($80k).

Your task is to provide a brief executive summary. Please format the output strictly as follows:
- **Total Revenue:** [Sum of all quarters]
- **QoQ Growth:** [Highlight the largest percentage transition between consecutive quarters]
- **Key Takeaway:** [One sentence summarizing the trend for next year's forecasting]"""

display(Markdown(f"**✅ After Output (Improved Prompt):**\n\n{get_groq_response(improved_prompt_3, system_prompt_3)}"))

**✅ After Output (Improved Prompt):**

**Executive Summary:**

- **Total Revenue:** $235,000 (sum of all quarters: Q1 + Q2 + Q3 + Q4)
- **QoQ Growth:** The largest percentage transition between consecutive quarters is between Q3 ($60k) and Q4 ($80k), with a growth rate of 33.33% (calculated as ((Q4 - Q3) / Q3) * 100).
- **Key Takeaway:** Our sales data shows a strong increase in the fourth quarter, indicating a potential peak in annual sales, suggesting a conservative forecasting approach for next year's Q4 sales.

## 4. Text Summarization: Losing Key Value

### Original Prompt
`Summarize the current state of renewable energy based on recent articles.`

### Issues
- **Hallucination Risk:** Without providing specific text to summarize, the AI relies on generic training data, which might be outdated or irrelevant.
- **Ambiguous Scope:** Doesn't define what aspects of renewable energy (cost, adoption, policy) to focus on.
- **Unstructured:** Output length and presentation format are entirely left to the model's discretion.

In [23]:
original_prompt_4 = "Summarize the current state of renewable energy based on recent articles."

display(Markdown(f"**❌ Before Output (Original Prompt):**\n\n{get_groq_response(original_prompt_4)}"))

**❌ Before Output (Original Prompt):**

Based on recent articles up to my cut-off knowledge date (December 2023), here's a summary of the current state of renewable energy:

**Solar Energy:**

1. **Cost reduction**: The cost of solar energy has continued to decline, making it more competitive with fossil fuels. According to a report by the International Energy Agency (IEA), the cost of solar energy has fallen by 70% over the past decade.
2. **Increasing adoption**: Solar energy has become a mainstream source of electricity, with countries like China, the US, and Germany leading the charge. In 2022, solar energy accounted for 8% of global electricity generation.
3. **Advancements in technology**: Research and development have led to more efficient solar panels, longer-lasting batteries, and improved energy storage systems.

**Wind Energy:**

1. **Continued growth**: Wind energy has become a significant contributor to global electricity generation, with the US, China, and Europe leading the way. In 2022, wind energy accounted for 10% of global electricity generation.
2. **Cost reduction**: The cost of wind energy has decreased significantly, making it more competitive with fossil fuels. According to a report by the Global Wind Energy Council (GWEC), the cost of wind energy has fallen by 50% over the past decade.
3. **Advancements in technology**: Research and development have led to more efficient wind turbines, longer-lasting blades, and improved energy storage systems.

**Hydrokinetic Energy:**

1. **Emerging sector**: Hydrokinetic energy, which harnesses energy from ocean tides and currents, is an emerging sector that has shown significant potential.
2. **Advancements in technology**: Research and development have led to more efficient hydrokinetic turbines, improved energy storage systems, and increased deployment of projects.

**Geothermal Energy:**

1. **Growing adoption**: Geothermal energy has gained popularity as a reliable and renewable source of electricity, particularly in regions with significant geothermal resources.
2. **Advancements in technology**: Research and development have led to more efficient geothermal plants, improved exploration techniques, and increased deployment of projects.

**Energy Storage:**

1. **Growing importance**: Energy storage has become a critical component of the transition to renewable energy, as it enables the stable and reliable integration of intermittent sources like solar and wind.
2. **Advancements in technology**: Research and development have led to more efficient batteries, improved energy storage systems, and increased deployment of projects.

**Policy and Regulation:**

1. **Increasing targets**: Many countries have set ambitious targets for renewable energy deployment, with some aiming for 100% renewable electricity by 2050.
2. **Net metering and grid support**: Governments and regulatory bodies have implemented policies to support the integration of renewable energy into the grid, including net metering and grid support mechanisms.

Overall, the current state of renewable energy is one of rapid growth, cost reduction, and technological advancements. As the world continues to transition towards a low-carbon economy, renewable energy is expected to play an increasingly important role in meeting energy demands while reducing greenhouse gas emissions.

--- 
### Improved Prompt
Adding a role, narrowing the scope visually, implementing negative constraints (exclude hydro/nuclear), and demanding a rigid structural format.
```text
Act as an Environmental Policy Researcher. 

Objective: Summarize the current trajectory of global solar and wind energy adoption. Focus strictly on cost reductions and governmental policy shifts over the last 3 years.

Format:
1. Executive Summary (2 sentences max)
2. Cost Reductions (3 bullet points)
3. Policy Shifts (3 bullet points)

Constraint: Do not include information about hydroelectric or nuclear energy. Keep the tone strictly academic.
```

In [24]:
system_prompt_4 = "Act as an Environmental Policy Researcher."
improved_prompt_4 = """Objective: Summarize the current trajectory of global solar and wind energy adoption. Focus strictly on cost reductions and governmental policy shifts over the last 3 years.

Format:
1. Executive Summary (2 sentences max)
2. Cost Reductions (3 bullet points)
3. Policy Shifts (3 bullet points)

Constraint: Do not include information about hydroelectric or nuclear energy. Keep the tone strictly academic."""

display(Markdown(f"**✅ After Output (Improved Prompt):**\n\n{get_groq_response(improved_prompt_4, system_prompt_4)}"))

**✅ After Output (Improved Prompt):**

**Executive Summary**

Global solar and wind energy adoption has experienced significant growth over the last three years, driven by substantial cost reductions and supportive governmental policies. This trend suggests a potential tipping point in the transition towards renewable energy, with solar and wind energy poised to become increasingly competitive with fossil fuels.

**Cost Reductions**

• **Solar Panel Costs**: The average cost of solar panels has declined by approximately 63% over the past three years, from $0.76 per watt in 2020 to $0.28 per watt in 2023, making solar energy a more viable option for widespread adoption.
• **Wind Turbine Costs**: The cost of wind turbines has also decreased, with the average cost per megawatt (MW) dropping by 45% between 2020 and 2023, from $1.36 million to $743,000.
• **Battery Storage Costs**: The cost of lithium-ion battery storage has decreased by 71% over the past three years, from $1,100 per kilowatt-hour (kWh) in 2020 to $315/kWh in 2023, enabling greater grid stability and efficiency for solar and wind energy.

**Policy Shifts**

• **Renewable Portfolio Standards (RPS)**: Many countries have strengthened their RPS policies, requiring utilities to generate a larger percentage of their electricity from renewable sources. For example, the European Union has increased its RPS target from 32% to 40% by 2030.
• **Tax Credits and Incentives**: Governments have continued to offer tax credits and incentives to support the development and deployment of solar and wind energy projects. The Inflation Reduction Act (IRA) in the United States, for instance, provides tax credits for solar and wind energy projects that begin construction before 2033.
• **Net Metering and Grid Connectivity**: Governments have implemented policies to improve net metering and grid connectivity for solar and wind energy, such as the U.S. Federal Energy Regulatory Commission's (FERC) Order 841, which aims to improve grid operations and enable greater participation from distributed energy resources.

## 5. Creative Ideation: The Blank Canvas

### Original Prompt
`Give me names for a coffee shop.`

### Issues
- **Lacks Vibe/Theme:** No indication of the shop's brand identity, location, or target audience.
- **No Limit on Generation:** The AI might spit out 50 names, overwhelming the user.
- **No Rationale:** Gives names without explaining why they work, which is critical for branding.

In [25]:
original_prompt_5 = "Give me names for a coffee shop."

display(Markdown(f"**❌ Before Output (Original Prompt):**\n\n{get_groq_response(original_prompt_5)}"))

**❌ Before Output (Original Prompt):**

Here are some name suggestions for a coffee shop:

1. **Brewed Awakening**: A playful name that evokes the idea of starting your day off right.
2. **The Cozy Cup**: A warm and inviting name that suggests a comfortable atmosphere.
3. **Java Joint**: A casual, laid-back name that's perfect for a relaxed coffee shop.
4. **The Daily Grind**: A clever name that pokes fun at the daily routine of a coffee shop.
5. **The Perk**: A fun, energetic name that suggests a lively atmosphere.
6. **The Coffee Club**: A name that suggests a sense of community and belonging.
7. **Bean Scene**: A playful name that references the coffee bean.
8. **The Roost**: A cozy, welcoming name that suggests a comfortable spot to relax.
9. **Cup & Co.**: A simple, elegant name that emphasizes the coffee cup.
10. **The Brew Haus**: A name that suggests a sense of tradition and expertise.
11. **The Coffee Parlor**: A classic, old-fashioned name that evokes a sense of nostalgia.
12. **Fuel & Flourish**: A name that suggests a coffee shop that's not just a place to grab a cup, but a hub for creativity and productivity.
13. **The Coffee Studio**: A name that suggests a creative, artistic atmosphere.
14. **The Daily Brew**: A name that emphasizes the idea of a fresh, daily cup of coffee.
15. **The Coffee Collective**: A name that suggests a sense of community and collaboration.

Remember, the name of your coffee shop should reflect your brand and values, so choose one that resonates with you and your target audience.

--- 
### Improved Prompt
Providing rich context, defining a specific output volume, and utilizing negative constraints.
```text
Act like a Brand Strategist. I am opening an artisanal coffee shop in a rainy city (Seattle). Our target demographic is remote workers and readers looking for a cozy, quiet place to stay for hours. 

Task: Generate exactly 4 distinct coffee shop name concepts. 
For each concept provide:
- The Name
- The Branding "Vibe" (1 sentence describing the visual aesthetic)
- Why it appeals to the target demographic.

Constraint: Avoid punny names (e.g., "Mug Shots") and generic words like "Cafe" or "Beans".
```

In [26]:
system_prompt_5 = "Act like a Brand Strategist."
improved_prompt_5 = """I am opening an artisanal coffee shop in a rainy city (Seattle). Our target demographic is remote workers and readers looking for a cozy, quiet place to stay for hours. 

Task: Generate exactly 4 distinct coffee shop name concepts. 
For each concept provide:
- The Name
- The Branding \"Vibe\" (1 sentence describing the visual aesthetic)
- Why it appeals to the target demographic.

Constraint: Avoid punny names (e.g., \"Mug Shots\") and generic words like \"Cafe\" or \"Beans\"."""

display(Markdown(f"**✅ After Output (Improved Prompt):**\n\n{get_groq_response(improved_prompt_5, system_prompt_5)}"))

**✅ After Output (Improved Prompt):**

Here are four distinct coffee shop name concepts that fit your target demographic:

**Concept 1:**
- **Name:** Haven & Page
- **Branding Vibe:** Warm, inviting lighting and plush armchairs in rich, earthy tones that evoke a sense of comfort and serenity.
- **Appeal to target demographic:** The name "Haven & Page" suggests a refuge for readers and remote workers, offering a space to escape the rain and focus on work or immerse themselves in a good book.

**Concept 2:**
- **Name:** Fable & Grounds
- **Branding Vibe:** Moody, atmospheric lighting and rich, dark wood accents that create a sense of mystery and intrigue.
- **Appeal to target demographic:** The name "Fable & Grounds" evokes a sense of storytelling and community, suggesting a space where people can gather to share ideas and connect over a cup of expertly crafted coffee.

**Concept 3:**
- **Name:** The Cozy Cup
- **Branding Vibe:** Soft, pastel colors and natural textures like woven baskets and reclaimed wood that create a sense of warmth and approachability.
- **Appeal to target demographic:** The name "The Cozy Cup" speaks directly to the desire for a welcoming, comforting space where remote workers and readers can settle in for hours.

**Concept 4:**
- **Name:** Ember & Oak
- **Branding Vibe:** Industrial-chic decor with exposed brick, metal accents, and a warm, fiery color palette that suggests energy and creativity.
- **Appeal to target demographic:** The name "Ember & Oak" suggests a space that's both inspiring and productive, where remote workers and readers can fuel their passions and pursue their creative endeavors.